In [7]:
import json
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# -------------------------------------------------------
# 1. LOAD DATASET (val)
# -------------------------------------------------------
def load_qa_file(json_path, label_path):
    # Load câu hỏi
    with open(json_path, "r", encoding="utf8") as f:
        qa_list = json.load(f)

    # Load nhãn
    with open(label_path, "r", encoding="utf8") as f:
        labels = json.load(f)

    X = []
    y = []

    for item in qa_list:
        qid = item["qid"]
        question = item["question"]
        y.append(int(labels[qid]))        # nhãn dạng string → int
        X.append(question)

    return X, y

# Load tập VAL để train
X_train, y_train = load_qa_file("./data/val.json", "./data/val_labels.json")    

# -------------------------------------------------------
# 2. DEFINE MODEL (Pipeline: TF-IDF + Linear SVM)
# -------------------------------------------------------
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features= 20000,
        sublinear_tf=True
    )),
    ("svm", LinearSVC())
])

# -------------------------------------------------------
# 3. TRAIN MODEL
# -------------------------------------------------------
print("Training model...")
model.fit(X_train, y_train)

# Lưu model sau khi train
joblib.dump(model, "svm_model.pkl")
print("Model đã được lưu vào svm_model.pkl")

# -------------------------------------------------------
# 4. LOAD TESTSET
# -------------------------------------------------------
X_test, y_test = load_qa_file("./data/test.json", "./data/test_labels.json")

# -------------------------------------------------------
# 5. PREDICT & EVALUATE
# -------------------------------------------------------
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Training model...
Model đã được lưu vào svm_model.pkl

Accuracy: 0.8837837837837837

Classification report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        80
           1       0.89      0.81      0.85       150
           2       0.82      0.89      0.85       140

    accuracy                           0.88       370
   macro avg       0.90      0.90      0.90       370
weighted avg       0.89      0.88      0.88       370

